# LLM-based Mistake Identification Analysis

This notebook implements a comprehensive pipeline for:
1. Loading the dataset from JSON files
2. Solving math problems using Mathstral LLM to generate answer keys
3. Evaluating mistake identification using Llama 3.1 70B
4. Analyzing results with confusion matrices and performance metrics

## Step 1: Import Required Libraries and Setup

In [ ]:
import json
import os
import re
from typing import Dict, List, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# For LLM API calls
import requests
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 2: Load Dataset from JSON Files

In [ ]:
# Define paths
DATA_DIR = "/DATA/cs24resch11011/repos/pedagogical-assessment/data"
TRAINSET_PATH = os.path.join(DATA_DIR, "trainset.json")
DEVSET_PATH = os.path.join(DATA_DIR, "dev_testset.json")
TESTSET_PATH = os.path.join(DATA_DIR, "testset.json")

# Load trainset
with open(TRAINSET_PATH, 'r') as f:
    trainset = json.load(f)

print(f"Loaded {len(trainset)} conversations from trainset.json")
print(f"\nFirst conversation structure:")
print(f"- conversation_id: {trainset[0]['conversation_id']}")
print(f"- conversation_history: {trainset[0]['conversation_history'][:200]}...")
print(f"- Number of tutor responses: {len(trainset[0]['tutor_responses'])}")
print(f"- Response sources: {list(trainset[0]['tutor_responses'].keys())}")

## Step 3: Extract Math Questions from Conversations

In [ ]:
def extract_question_from_conversation(conversation_history: str) -> str:
    """
    Extract the math question from the conversation history.
    The question usually appears after "The question is:" in the first tutor message.
    """
    # Look for pattern "The question is:" or "question below?"
    patterns = [
        r"The question is:\s*(.+?)\s*\n\s*Student:",
        r"question below\?\s*(.+?)\s*\n\s*Student:",
        r"Tutor:\s*(.+?)\s*\n\s*Student:"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, conversation_history, re.DOTALL)
        if match:
            question = match.group(1).strip()
            # Clean up the question
            question = question.replace("\\n", " ").strip()
            if len(question) > 20:  # Valid question should be reasonably long
                return question
    
    # Fallback: extract first tutor message
    lines = conversation_history.split("\n")
    for line in lines:
        if line.strip().startswith("Tutor:"):
            return line.replace("Tutor:", "").strip()
    
    return conversation_history[:500]  # Last resort

# Test extraction on first few conversations
print("Extracting questions from conversations...\n")
for i in range(min(3, len(trainset))):
    question = extract_question_from_conversation(trainset[i]['conversation_history'])
    print(f"Conversation {i+1}:")
    print(f"Question: {question[:200]}...")
    print()

## Step 4: Setup Mathstral LLM for Answer Generation

We'll use Mistral AI's Mathstral model (mathstral-7B-v0.1) for solving math problems.

In [ ]:
class MathSolver:
    """
    Math problem solver using Mathstral or similar math-focused LLM.
    Supports both local model loading and API-based inference.
    """
    
    def __init__(self, model_name="mistralai/Mathstral-7B-v0.1", use_local=True):
        self.model_name = model_name
        self.use_local = use_local
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        if use_local:
            print(f"Loading {model_name} model...")
            print(f"Using device: {self.device}")
            try:
                self.tokenizer = AutoTokenizer.from_pretrained(model_name)
                self.model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                    device_map="auto" if torch.cuda.is_available() else None,
                    low_cpu_mem_usage=True
                )
                print("Model loaded successfully!")
            except Exception as e:
                print(f"Error loading model: {e}")
                print("Falling back to API mode or alternative approach")
                self.use_local = False
    
    def solve_math_problem(self, question: str, max_length=1024) -> str:
        """
        Solve a math problem and return the step-by-step solution with final answer.
        """
        prompt = f"""Solve the following math problem step by step. Provide a clear, detailed solution and state the final answer clearly at the end.

Question: {question}

Solution:"""
        
        if self.use_local and hasattr(self, 'model'):
            try:
                inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
                
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=max_length,
                        temperature=0.1,  # Low temperature for more deterministic math answers
                        do_sample=True,
                        top_p=0.9,
                        pad_token_id=self.tokenizer.eos_token_id
                    )
                
                solution = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                # Extract only the solution part (after the prompt)
                solution = solution.split("Solution:")[-1].strip()
                return solution
            except Exception as e:
                print(f"Error during inference: {e}")
                return self._fallback_solution(question)
        else:
            return self._fallback_solution(question)
    
    def _fallback_solution(self, question: str) -> str:
        """
        Fallback method when model is not available.
        Returns a placeholder that indicates manual solution is needed.
        """
        return f"[Solution requires computation for: {question[:100]}...]"

# Initialize the math solver
print("Initializing Math Solver...")
math_solver = MathSolver(model_name="mistralai/Mathstral-7B-v0.1", use_local=True)
print("Math Solver ready!")

## Step 5: Generate Answers for All Questions

Generate answer keys for all conversations in the trainset.

In [ ]:
from tqdm import tqdm

def generate_answers_for_dataset(dataset, math_solver, max_samples=None):
    """
    Generate answer keys for all conversations in the dataset.
    
    Args:
        dataset: List of conversation dictionaries
        math_solver: MathSolver instance
        max_samples: Maximum number of samples to process (None for all)
    
    Returns:
        Updated dataset with 'answer_key' field added
    """
    dataset_with_answers = []
    
    num_samples = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    
    print(f"Generating answers for {num_samples} conversations...")
    
    for i, conversation in enumerate(tqdm(dataset[:num_samples], desc="Solving problems")):
        # Create a copy of the conversation
        conv_with_answer = conversation.copy()
        
        # Extract question
        question = extract_question_from_conversation(conversation['conversation_history'])
        
        # Generate answer
        try:
            answer = math_solver.solve_math_problem(question)
            conv_with_answer['answer_key'] = answer
        except Exception as e:
            print(f"\nError solving question {i+1}: {e}")
            conv_with_answer['answer_key'] = f"[Error generating answer: {str(e)}]"
        
        dataset_with_answers.append(conv_with_answer)
        
        # Show progress for first few
        if i < 3:
            print(f"\n{'='*80}")
            print(f"Conversation {i+1}:")
            print(f"Question: {question[:150]}...")
            print(f"Answer: {conv_with_answer['answer_key'][:200]}...")
    
    return dataset_with_answers

# Generate answers for a subset first (for testing)
# Set to None to process all conversations
NUM_SAMPLES_TO_PROCESS = 10  # Start with 10 for testing, change to None for full dataset

trainset_with_answers = generate_answers_for_dataset(
    trainset, 
    math_solver, 
    max_samples=NUM_SAMPLES_TO_PROCESS
)

print(f"\n{'='*80}")
print(f"Successfully generated answers for {len(trainset_with_answers)} conversations!")

## Step 6: Save Updated Dataset with Answer Keys

In [ ]:
# Save the updated dataset
OUTPUT_DIR = "/DATA/cs24resch11011/repos/pedagogical-assessment/devendra/mistake_identification/llm_approach"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DIR, "trainset_with_answers.json")

with open(OUTPUT_FILE, 'w') as f:
    json.dump(trainset_with_answers, f, indent=2)

print(f"Saved updated dataset to: {OUTPUT_FILE}")
print(f"Total conversations: {len(trainset_with_answers)}")
print(f"\nSample conversation structure:")
print(json.dumps(trainset_with_answers[0], indent=2)[:1000] + "...")

## Step 7: Setup Llama 3.1 70B for Mistake Identification Evaluation

We'll use Llama 3.1 70B (or 8B if 70B is not available) to evaluate whether the tutor responses correctly identified the student's mistake.

In [ ]:
class MistakeIdentificationEvaluator:
    """
    Evaluator using Llama 3.1 70B (or 8B) to assess whether tutor responses
    correctly identified student mistakes.
    """
    
    def __init__(self, model_name="meta-llama/Llama-3.1-8B-Instruct", use_local=True):
        """
        Initialize the evaluator.
        Using 8B version as it's more accessible. Can change to 70B if available:
        "meta-llama/Llama-3.1-70B-Instruct"
        """
        self.model_name = model_name
        self.use_local = use_local
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        if use_local:
            print(f"Loading {model_name} model...")
            print(f"Using device: {self.device}")
            try:
                self.tokenizer = AutoTokenizer.from_pretrained(model_name)
                self.model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                    device_map="auto" if torch.cuda.is_available() else None,
                    low_cpu_mem_usage=True
                )
                print("Model loaded successfully!")
            except Exception as e:
                print(f"Error loading model: {e}")
                print("Will use rule-based fallback")
                self.use_local = False
    
    def evaluate_mistake_identification(
        self, 
        conversation_history: str, 
        correct_answer: str, 
        tutor_response: str
    ) -> str:
        """
        Evaluate whether the tutor response correctly identified the student's mistake.
        
        Returns: "Yes", "No", or "To some extent"
        """
        prompt = f"""You are an expert educational assessment evaluator. Your task is to evaluate whether a tutor correctly identified a student's mistake in their response.

You will be given:
1. The conversation history between tutor and student
2. The correct answer to the problem
3. The tutor's response to the student

Based on this information, determine if the tutor successfully identified the student's mistake. Respond with EXACTLY ONE of these three options:
- "Yes" - if the tutor clearly and correctly identified the student's mistake
- "No" - if the tutor failed to identify the mistake or identified the wrong issue
- "To some extent" - if the tutor partially identified the mistake but missed key aspects or was unclear

CONVERSATION HISTORY:
{conversation_history}

CORRECT ANSWER:
{correct_answer}

TUTOR'S RESPONSE:
{tutor_response}

EVALUATION (respond with ONLY "Yes", "No", or "To some extent"):"""

        if self.use_local and hasattr(self, 'model'):
            try:
                inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(self.device)
                
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=50,
                        temperature=0.1,
                        do_sample=True,
                        top_p=0.9,
                        pad_token_id=self.tokenizer.eos_token_id
                    )
                
                response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                # Extract only the evaluation part
                response = response.split("EVALUATION")[-1].strip()
                
                # Parse the response to get clean category
                return self._parse_evaluation(response)
            except Exception as e:
                print(f"Error during evaluation: {e}")
                return self._fallback_evaluation(conversation_history, tutor_response)
        else:
            return self._fallback_evaluation(conversation_history, tutor_response)
    
    def _parse_evaluation(self, response: str) -> str:
        """
        Parse the model's response to extract the evaluation category.
        """
        response_lower = response.lower()
        
        # Check for each category
        if "to some extent" in response_lower:
            return "To some extent"
        elif any(word in response_lower for word in ["yes", "correct", "successfully"]):
            # Make sure it's not "no" or negative
            if not any(word in response_lower for word in ["no", "not", "fail", "incorrect"]):
                return "Yes"
        
        # Check for "No"
        if any(word in response_lower[:50] for word in ["no", "not", "fail", "incorrect", "did not"]):
            return "No"
        
        # Default to "To some extent" if unclear
        return "To some extent"
    
    def _fallback_evaluation(self, conversation_history: str, tutor_response: str) -> str:
        """
        Simple rule-based fallback when model is not available.
        """
        response_lower = tutor_response.lower()
        
        # Keywords indicating mistake identification
        mistake_keywords = [
            "mistake", "error", "incorrect", "wrong", "not quite",
            "let's reconsider", "actually", "remember", "should be"
        ]
        
        guidance_keywords = [
            "try", "can you", "what if", "let's", "consider",
            "think about", "recall", "remember that"
        ]
        
        mistake_count = sum(1 for keyword in mistake_keywords if keyword in response_lower)
        guidance_count = sum(1 for keyword in guidance_keywords if keyword in response_lower)
        
        if mistake_count >= 2:
            return "Yes"
        elif mistake_count >= 1 or guidance_count >= 2:
            return "To some extent"
        else:
            return "No"

# Initialize the evaluator
print("Initializing Mistake Identification Evaluator...")
evaluator = MistakeIdentificationEvaluator(
    model_name="meta-llama/Llama-3.1-8B-Instruct",  # Change to 70B if available
    use_local=True
)
print("Evaluator ready!")

## Step 8: Evaluate Mistake Identification for All Tutor Responses

Evaluate each tutor response (from different LLMs and experts) to determine if they correctly identified the student's mistake.

In [ ]:
def evaluate_all_responses(dataset_with_answers, evaluator):
    """
    Evaluate mistake identification for all tutor responses in the dataset.
    Adds 'llm_predicted_mistake_identification' to each response's annotation.
    """
    evaluated_dataset = []
    
    print(f"Evaluating mistake identification for all conversations...")
    
    for conv_idx, conversation in enumerate(tqdm(dataset_with_answers, desc="Evaluating")):
        evaluated_conv = conversation.copy()
        
        # Get conversation details
        conversation_history = conversation['conversation_history']
        answer_key = conversation.get('answer_key', 'Not available')
        
        # Evaluate each tutor response
        for source_name, response_data in conversation['tutor_responses'].items():
            tutor_response = response_data['response']
            
            # Perform evaluation
            try:
                predicted_category = evaluator.evaluate_mistake_identification(
                    conversation_history,
                    answer_key,
                    tutor_response
                )
                
                # Add prediction to annotation
                if 'annotation' not in evaluated_conv['tutor_responses'][source_name]:
                    evaluated_conv['tutor_responses'][source_name]['annotation'] = {}
                
                evaluated_conv['tutor_responses'][source_name]['annotation']['LLM_Predicted_Mistake_Identification'] = predicted_category
                
            except Exception as e:
                print(f"\nError evaluating conversation {conv_idx+1}, source {source_name}: {e}")
                evaluated_conv['tutor_responses'][source_name]['annotation']['LLM_Predicted_Mistake_Identification'] = "Error"
        
        evaluated_dataset.append(evaluated_conv)
        
        # Show first few examples
        if conv_idx < 2:
            print(f"\n{'='*80}")
            print(f"Conversation {conv_idx+1}:")
            for source_name, response_data in evaluated_conv['tutor_responses'].items():
                actual = response_data['annotation'].get('Mistake_Identification', 'N/A')
                predicted = response_data['annotation'].get('LLM_Predicted_Mistake_Identification', 'N/A')
                print(f"  {source_name}:")
                print(f"    Actual: {actual}")
                print(f"    Predicted: {predicted}")
    
    return evaluated_dataset

# Perform evaluation
trainset_evaluated = evaluate_all_responses(trainset_with_answers, evaluator)

print(f"\n{'='*80}")
print(f"Evaluation complete for {len(trainset_evaluated)} conversations!")

## Step 9: Save Final Dataset with Evaluations

In [ ]:
# Save the final evaluated dataset
FINAL_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "trainset_with_answers_and_evaluations.json")

with open(FINAL_OUTPUT_FILE, 'w') as f:
    json.dump(trainset_evaluated, f, indent=2)

print(f"Saved final evaluated dataset to: {FINAL_OUTPUT_FILE}")
print(f"Total conversations: {len(trainset_evaluated)}")
print(f"\nDataset now includes:")
print("  - Original conversation history")
print("  - Answer keys generated by Mathstral")
print("  - Tutor responses from various sources")
print("  - Original annotations")
print("  - LLM-predicted mistake identification categories")

## Step 10: Prepare Data for Analysis

Extract predictions and actual labels for comparison.

In [ ]:
def prepare_analysis_data(evaluated_dataset):
    """
    Prepare data for analysis by extracting actual vs predicted labels.
    
    Returns:
        DataFrame with columns: conversation_id, source, actual, predicted
    """
    records = []
    
    for conversation in evaluated_dataset:
        conv_id = conversation['conversation_id']
        
        for source_name, response_data in conversation['tutor_responses'].items():
            annotation = response_data.get('annotation', {})
            
            actual = annotation.get('Mistake_Identification', 'Unknown')
            predicted = annotation.get('LLM_Predicted_Mistake_Identification', 'Unknown')
            
            # Skip if either is unknown or error
            if actual != 'Unknown' and predicted not in ['Unknown', 'Error']:
                records.append({
                    'conversation_id': conv_id,
                    'source': source_name,
                    'actual': actual,
                    'predicted': predicted,
                    'response': response_data['response']
                })
    
    df = pd.DataFrame(records)
    return df

# Prepare data
analysis_df = prepare_analysis_data(trainset_evaluated)

print(f"Analysis DataFrame shape: {analysis_df.shape}")
print(f"\nFirst few rows:")
print(analysis_df.head(10))
print(f"\nActual label distribution:")
print(analysis_df['actual'].value_counts())
print(f"\nPredicted label distribution:")
print(analysis_df['predicted'].value_counts())
print(f"\nSource distribution:")
print(analysis_df['source'].value_counts())

## Step 11: Generate Confusion Matrix

Create confusion matrices comparing predicted vs actual mistake identification.